# Lab 10: Correlation & Causality — Which Story Made This Scatterplot?
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 80 min Core + 30 min Write it yourself + 15 min Extension + take-home Challenge

---

**Learning Objectives:**
- Show that causation, confounding and selection can leave the same correlation in the data, and explain why no test of that correlation separates them
- Encode a causal claim as a directed graph and read the paths off it
- Demonstrate that controlling for a confounder removes bias while controlling for a collider creates it
- Explain why the "control for anything pre-treatment" rule fails
- Write a small regression function, and use it to close a path that a bad control opened

**Dataset:** Simulated DGPs with known truth, plus the LaLonde NSW job-training experiment and its CPS comparison group

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one.

**How this lab works:** Parts 1-3 are GUIDED: run each cell as it is, then answer the questions. Part 4 has three one-line blanks in two cells, with one guided step between them. "Write it yourself" is required: a short function you write from a blank cell. The Extension is optional, and the Challenge is a take-home.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 4, Steps 8 and 10:** three one-line blanks (`____`). In Step 8, the outcome `y` of the M-bias world, and the regression that controls for W, modelled on Step 5. In Step 10, the share of the LaLonde gap that the controls closed.
> - **Write it yourself (required),** after Part 4: `coef_on_d()`, a function that regresses an outcome on D plus a list of controls and gives back D's coefficient, then one call to it on Step 8's data and a print. About five lines from a blank cell. The check cell below it must print "Passed". Then one or two sentences of text on why the result comes out as it does.
> - The answers to the interpretation questions after each part, in text cells.
>
> **You run and read:** everything else. Setup, Parts 1-3, Step 9 and the Extension's simulation are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The Extension (optional) asks you to change two numbers in the simulation, run it again, and answer in two sentences.
>
> **The Challenge** (take-home) has no starter code: you write it from the specification, in new cells. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

Every random number in this lab comes from one generator, `rng`, and each draw moves it on. So
your numbers match the expected output only if you run the cells once, from the top, in order. If
you run a cell twice, use **Runtime → Restart session** and run from the top again.

In [ ]:
# GUIDED — run as-is
# Step 1: imports and the random generator
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import statsmodels.api as sm
from pathlib import Path

rng = np.random.default_rng(seed=42)    # every simulated draw below comes from rng, in order
RANDOM_STATE = 42                       # fixes where networkx places the nodes in Step 4

# savefig will not create a folder, and a fresh Colab session has no figures/ folder
Path("figures").mkdir(exist_ok=True)

print("Setup complete.")

## Part 1: Three Worlds, One Scatterplot (GUIDED — 15 min)

Chapter 10 opens with a claim worth taking seriously: causation, confounding and selection can
leave the **same correlation** in the data, and the data alone cannot tell you which one made it.
Below are three data-generating processes where you know the truth because you wrote it.

`np.corrcoef(d, y)` gives a 2 x 2 table of correlations; `[0, 1]` picks the one between `d` and `y`.

In [ ]:
# GUIDED — run as-is
# Step 2: build three worlds with very different truths
n = 5_000          # the underscore only makes 5000 easier to read

# World 1: D causes Y, effect = 2.0. The large noise pulls the correlation down to about
# the other two worlds' while the effect stays exactly 2.0
d1 = rng.normal(0, 1, n)
y1 = 2 * d1 + rng.normal(0, 2.2, n)

# World 2: a confounder U drives both. There is no d2 in the y2 line, so D has NO effect on Y
u = rng.normal(0, 1, n)
d2 = 1.4 * u + rng.normal(0, 1, n)
y2 = 1.4 * u + rng.normal(0, 1, n)

# World 3: D and Y are independent, but we only see units whose D exceeds Y by more than 1.2
d3_all = rng.normal(0, 1, n * 4)
y3_all = rng.normal(0, 1, n * 4)
# seen is True or False for each unit; d3_all[seen] keeps the units where it is True
seen = (d3_all - y3_all) > 1.2
d3, y3 = d3_all[seen], y3_all[seen]

# Each item in the list is (label, d, y, truth); the for line unpacks the four into names
for label, d, y, truth in [("1  causation ", d1, y1, 2.0),
                           ("2  confounding", d2, y2, 0.0),
                           ("3  selection ", d3, y3, 0.0)]:
    r = np.corrcoef(d, y)[0, 1]
    # " .3f" leaves a space where a minus sign would go, so the columns line up
    print(f"World {label}: corr(D, Y) = {r: .3f}   true effect = {truth:.1f}")

**Expected output:** correlations of 0.654, 0.664 and 0.632, with true effects of 2.0, 0.0 and 0.0.

In [ ]:
# GUIDED — run as-is
# Step 3: plot all three. Chapter 3 said "look at your data": try it

# zip pairs each panel with one world; sharex and sharey give all three panels the same axes
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
for ax, (title, d, y) in zip(axes, [("World 1: D causes Y", d1, y1),
                                    ("World 2: U causes both", d2, y2),
                                    ("World 3: selected sample", d3, y3)]):
    ax.scatter(d, y, s=4, alpha=0.25, color="darkslategray")
    ax.set_title(title)
    ax.set_xlabel("D")
axes[0].set_ylabel("Y")
plt.tight_layout()
plt.savefig("figures/ch10_three_worlds.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** three upward-sloping clouds, with correlations all between 0.63 and 0.67. In
one the effect is 2.0; in the other two it is exactly zero. World 3's cloud has a straight upper-left
edge, which Worlds 1 and 2 do not have, and nothing in the picture says why it is there. Anscombe's Quartet could be
solved by plotting. This cannot: what separates the worlds, U or the selection rule, is not a
column in the data.

### Interpretation Questions

1. All three correlations are large, and with thousands of points each one is highly significant. Explain in one sentence why statistical significance is irrelevant to the question being asked here.
2. World 2's estimate is biased. Does collecting 5 million observations instead of 5,000 help? Say what it *does* do.
3. World 3 contains no causal arrow at all, yet the correlation is strong. Name a real selection rule (who gets into a dataset) that would produce World 3, and say which sign of correlation it would create.

*Your answers here:*

1. 
2. 
3. 

## Part 2: Writing the Claim Down (GUIDED — 20 min)

Since the data cannot settle it, the structure has to be stated as an assumption, which other
people can then dispute. A directed acyclic graph (DAG) writes that assumption down as arrows, so
that its consequences can be worked out.

In [ ]:
# GUIDED — run as-is
# Step 4: draw the three canonical structures

# Each structure is a list of arrows, written (from, to); \n in a title starts a new line
structures = {
    "Confounder\nU -> D, U -> Y": [("U", "D"), ("U", "Y"), ("D", "Y")],
    "Collider\nD -> C <- Y":      [("D", "C"), ("Y", "C")],
    "Mediator\nD -> M -> Y":      [("D", "M"), ("M", "Y")],
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
# .items() gives (title, arrows) pairs, and zip pairs each one with a panel
for ax, (title, edges) in zip(axes, structures.items()):
    G = nx.DiGraph(edges)                           # a directed graph built from the arrows
    pos = nx.spring_layout(G, seed=RANDOM_STATE)    # where each node sits on the page
    nx.draw_networkx(G, pos, ax=ax, node_color="lightsteelblue", node_size=1400, arrowsize=18)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.savefig("figures/ch10_three_structures.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** read each graph by its arrowheads, not by where the nodes sit; the layout
places them wherever it likes. Same three letters, three different claims about the world, and
three different answers to "should I control for it?"

Step 5 simulates the confounder graph and runs two regressions. `sm.OLS(y, X)` takes the outcome first, then the table of regressors; `.fit()`
runs the regression and returns the results, whose `.params` and `.pvalues` you read.
`sm.add_constant(d)` puts a column
of 1s beside `d`, so the regression has an intercept; in `.params`, index 0 is the intercept and
index 1 is the coefficient on D. `np.column_stack([d, u])` sets `d` and `u` side by side as two
columns, which is how a control enters the regression.

In [ ]:
# GUIDED — run as-is
# Step 5: a good control. U confounds; adjusting for it recovers the truth
n = 10_000
u = rng.normal(0, 1, n)
d = 0.8 * u + rng.normal(0, 1, n)
y = 1.5 * d + 2.0 * u + rng.normal(0, 1, n)     # the TRUE effect of D on Y is 1.5

naive = sm.OLS(y, sm.add_constant(d)).fit()
adjusted = sm.OLS(y, sm.add_constant(np.column_stack([d, u]))).fit()

# <26 pads the text to 26 characters; >10.3f right-aligns a number in 10, with 3 decimals
print(f"{'specification':<26}{'estimate':>10}{'truth':>8}")
print("-" * 44)
print(f"{'Y ~ D':<26}{naive.params[1]:>10.3f}{1.5:>8.1f}")
print(f"{'Y ~ D + U  (adjusted)':<26}{adjusted.params[1]:>10.3f}{1.5:>8.1f}")

**Expected output:** Y on D alone gives 2.476, against a true effect of 1.5. With U as a control,
the estimate is 1.500: controlling for the confounder put it on the truth. This is the case
everyone has in mind when they say "add controls".

### Interpretation Questions

1. In Step 5 the naive estimate is too *large*. Work out from the DGP why the bias is positive, and say what would make it negative.
2. The adjusted regression recovers 1.5 because we measured `u`. What is the argument you would have to make in a real paper, where `u` is something like "motivation"?
3. A DAG has no numbers in it. What exactly does it give you, if it cannot tell you the size of an effect?

*Your answers here:*

1. 
2. 
3. 

## Part 3: The Control That Creates Bias (GUIDED — 15 min)

Controlling for C creates an association between D and Y, which are independent by
construction. The code is nearly identical to Step 5's; only the variable you control for changes.

In [ ]:
# GUIDED — run as-is
# Step 6: a collider. D and Y are independent by construction
n = 10_000
d = rng.normal(0, 1, n)
y = rng.normal(0, 1, n)                         # NO arrow between D and Y
c = 1.2 * d + 1.2 * y + rng.normal(0, 1, n)     # both cause C

no_control = sm.OLS(y, sm.add_constant(d)).fit()
with_c = sm.OLS(y, sm.add_constant(np.column_stack([d, c]))).fit()

print(f"{'specification':<26}{'estimate':>10}{'p':>10}{'truth':>8}")
print("-" * 54)
print(f"{'Y ~ D':<26}{no_control.params[1]:>10.3f}{no_control.pvalues[1]:>10.3f}{0.0:>8.1f}")
print(f"{'Y ~ D + C  (collider!)':<26}{with_c.params[1]:>10.3f}{with_c.pvalues[1]:>10.3f}{0.0:>8.1f}")

**Expected output:** without controls, −0.017 with p = 0.090: close to the truth of zero. With C as
a control, −0.592, and a p-value that prints as 0.000, meaning below 0.0005. The truth is zero in
both rows, so the second estimate is an effect that does not exist, with a p-value that says it
is very unlikely to be chance.

In [ ]:
# GUIDED — run as-is
# Step 7: why it happens. Look inside a slice of the collider

# band is True for the units whose C is within 0.35 of its median: C held roughly fixed
band = np.abs(c - np.median(c)) < 0.35
corr_all = np.corrcoef(d, y)[0, 1]
corr_band = np.corrcoef(d[band], y[band])[0, 1]

# True counts as 1 in a sum, so band.sum() counts the units in the band; :, adds thousands commas
print(f"Units in the band: {band.sum():,} of {n:,}")
print(f"corr(D, Y), everyone:        {corr_all:.3f}")
print(f"corr(D, Y), inside the band: {corr_band:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
axes[0].scatter(d, y, s=4, alpha=0.15, color="darkslategray")
axes[0].set_title(f"Everyone: corr = {corr_all:.3f}")
axes[1].scatter(d[band], y[band], s=6, alpha=0.5, color="firebrick")
axes[1].set_title(f"Holding C fixed: corr = {corr_band:.3f}")
for ax in axes:
    ax.set_xlabel("D")
axes[0].set_ylabel("Y")
plt.tight_layout()
plt.savefig("figures/ch10_collider_slice.png", dpi=150, bbox_inches="tight")
plt.show()

**Expected output:** 1,366 of the 10,000 units are in the band. Across everyone, corr(D, Y) is
−0.017; inside the band it is −0.596. Inside a slice of C, a high D must be offset by a low Y to
keep C the same. That trade-off is the manufactured correlation. Nothing about the world changed,
only which units we are comparing.

### Interpretation Questions

1. Section 10.4's restaurant example (good food *or* convenient location) is the same structure. Map "food", "location" and "worth visiting" onto D, Y and C.
2. Chapter 6 called this selection bias; Chapter 10 calls it collider bias. Explain why "your dataset is the units that were selected" makes them the same thing.
3. You are handed a regression with a significant coefficient and a long list of controls. What single question do you now know to ask?

*Your answers here:*

1. 
2. 
3. 

## Part 4: M-Bias and the LaLonde Debate (YOUR TASK — 30 min)

Most practitioners protect themselves with a rule: *control for anything measured before
treatment*. It is a decent screen against mediators. It is not sufficient, and M-bias is why.

In Step 8, two unmeasured causes, u1 and u2, both feed into W. u1 also drives the treatment D,
and u2 drives the outcome Y. Drawn out, the path is D ← u1 → W ← u2 → Y, an M shape, with W a
collider in the middle.

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Step 8: W is PRE-TREATMENT, not caused by D or Y, and still a bad control
n = 20_000

# Two unmeasured causes: u1 drives treatment, u2 drives the outcome
u1 = rng.normal(0, 1, n)
u2 = rng.normal(0, 1, n)

# W is a common effect of u1 and u2 (a collider), measured before treatment
w = 1.0 * u1 + 1.0 * u2 + rng.normal(0, 1, n)

d = 1.0 * u1 + rng.normal(0, 1, n)

# Blank 1: y is built from u2 the way d is built from u1, and does NOT depend on d,
# so the true effect of D on Y is exactly zero
y = ____

without_w = sm.OLS(y, sm.add_constant(d)).fit()
# Blank 2: the same regression with W as a control beside D, as Step 5's adjusted regression does with U
with_w = ____

print(f"{'specification':<30}{'estimate':>10}{'truth':>8}")
print("-" * 48)
print(f"{'Y ~ D':<30}{without_w.params[1]:>10.3f}{0.0:>8.1f}")
print(f"{'Y ~ D + W  (pre-treatment!)':<30}{with_w.params[1]:>10.3f}{0.0:>8.1f}")

**Expected output:** 0.009 without W, and −0.191 with W, where the truth is zero in both rows. If
your numbers differ, check Blank 1 against the `d` line, and run the notebook from the top: every
draw from `rng` moves it on. *NameError: name '____' is not defined* means a blank is still empty.

In [ ]:
# GUIDED — run as-is
# Step 9: the LaLonde debate. First, the experiment

# Dehejia & Wahba (2002) files, served by NBER; .dta is a Stata file.
# nsw_dw.dta is the randomised experiment; cps_controls.dta is a comparison group from the CPS survey
EXPERIMENT = "https://users.nber.org/~rdehejia/data/nsw_dw.dta"
CPS = "https://users.nber.org/~rdehejia/data/cps_controls.dta"
exp_df = pd.read_stata(EXPERIMENT)
cps_df = pd.read_stata(CPS)

# treat is 1 for a trainee and 0 for a control; re78 is 1978 earnings, the outcome.
# .loc[rows, column] picks the re78 values of the rows where the condition is True
treated_earnings = exp_df.loc[exp_df["treat"] == 1, "re78"]
control_earnings = exp_df.loc[exp_df["treat"] == 0, "re78"]

# Random assignment makes the simple difference in means the causal effect: the number to beat
benchmark = treated_earnings.mean() - control_earnings.mean()

print(f"Experimental groups: {len(treated_earnings)} treated, {len(control_earnings)} control")
print(f"Experimental benchmark (randomised): ${benchmark:,.0f}")

**Expected output:** 185 treated and 260 controls, and a benchmark of \$1,794, the experimental
effect you met in Lab 9.

Now throw the experimental controls away and compare the same trainees with the CPS instead. That
is what you would have to do if no experiment had been run.

In [ ]:
# GUIDED — run as-is
# Step 9, continued: the same trainees against a non-experimental comparison group

# Stack the trainees on top of the CPS rows (whose treat is 0); ignore_index numbers the rows afresh
obs = pd.concat([exp_df[exp_df["treat"] == 1], cps_df], ignore_index=True)
naive_obs = (obs.loc[obs["treat"] == 1, "re78"].mean()
             - obs.loc[obs["treat"] == 0, "re78"].mean())

print(f"Observational comparison group: {len(cps_df):,} CPS respondents")
print(f"Naive observational estimate:  ${naive_obs:,.0f}")

**Expected output:** 15,992 CPS respondents, and a naive estimate of −\$8,498 (it prints as
\$-8,498). Same treated people, same outcome: the comparison group changed, and the answer changed
sign. This is the same contrast LaLonde (1986) ran, here on Dehejia and Wahba's subsample, to ask whether
observational methods can recover experimental answers.

In [ ]:
# YOUR TASK — fill in the blank (____)
# Step 10: Adjust the observational estimate for observed covariates
covariates = ["age", "education", "black", "hispanic", "married",
              "nodegree", "re74", "re75"]

# + joins two lists: ["treat"] + covariates is one list of nine names.
# astype(float) turns every column into a plain number, which statsmodels needs.
# With a DataFrame, .params is labelled by column name, so the trainee effect is params["treat"]
X = sm.add_constant(obs[["treat"] + covariates].astype(float))
adjusted_obs = sm.OLS(obs["re78"].astype(float), X).fit()

# >9,.0f right-aligns in 9 characters, with thousands commas and no decimals, as in Step 7
print(f"Experimental benchmark:        ${benchmark:>9,.0f}")
print(f"Observational, no controls:    ${naive_obs:>9,.0f}")
print(f"Observational, with controls:  ${adjusted_obs.params['treat']:>9,.0f}")

# Blank: how far the controls moved the estimate, as a share of the whole distance
# from the naive estimate to the experimental benchmark
gap_closed = ____
# .0% multiplies by 100, drops the decimals and adds a percent sign
print(f"\nShare of the gap closed by controls: {gap_closed:.0%}")

**Expected output:** with the eight covariates, the estimate is \$699, and the controls closed 89%
of the gap. *NameError: name '____' is not defined* means the blank is still empty.

### Interpretation Questions

1. Your M-bias regression should show a clear effect where the truth is zero, using a control measured before treatment. State the rule this kills, and what replaces it.
2. The covariate adjustment moves the LaLonde estimate a long way. Does getting closer to the benchmark prove the adjustment is valid? What would the same result look like if you had no benchmark to check against — which is the normal situation?
3. The chapter warns that a control can be a confounder in one DAG and a collider in another. Pick one LaLonde covariate and describe a structure in which controlling for it would be a mistake.

*Your answers here:*

1. 
2. 
3. 

---

## Write it yourself (required — 30 min)

This part is required, and it is short. The check cell below it must print "Passed" before you
submit.

**The task.** In Step 8, controlling for W opened the path D ← u1 → W ← u2 → Y. In a simulation, u1 is not hidden from us. So what
does the estimate become if you control for W *and* u1?

Steps 5, 6 and 8 each ran the same regression with a different set of controls. Turn that into a
function, then use it.

Write a function `coef_on_d(y, d, controls)` that regresses `y` on `d` plus every array in the
list `controls`, with a constant, and gives back the coefficient on `d` alone. With an empty list,
`coef_on_d(y, d, [])` is Step 8's `Y ~ D`. Then call it on Step 8's arrays with W and u1 as the
controls, store the result as `m_bias_blocked`, and print it to 3 decimals.

- It takes about five lines: the `def` line, a one-line docstring, a line or two inside, the call, and a print.
- Step 5's adjusted regression has the shape you need. Here the controls arrive as a list, and two lists join into one with `+`; put `d` in a list of its own, `[d]`, first.
- In `.params`, index 0 is the constant. Give back one number, not the whole list.
- Lab 9's `n_needed()` shows the shape of a function: the `def` line, a one-line docstring, the body, and `return`.

Then run the check cell. It calls your function with no controls and with W alone, and compares
the answers with Step 8's; it also tries it on changed copies of `y`. Then it checks
`m_bias_blocked`. `assert` stops with its message when its condition is False; the message tells
you what to fix.

**Last, in the text cell under the check:** in one or two sentences, why does adding u1 undo the
damage W did? Name the path, say what role u1 plays on it, and say why you could not do this in
the LaLonde data.

In [ ]:
# WRITE IT YOURSELF (required): coef_on_d(), then m_bias_blocked for Step 8's data

In [ ]:
# CHECK — for "Write it yourself" above

# globals() holds every name defined so far in this notebook
if "coef_on_d" not in globals():
    print("Not written yet: write coef_on_d() in the cell above and run it, then run this check.")
else:
    # try/except catches an error inside your function, so the message can say where to look
    try:
        none_yours = coef_on_d(y, d, [])
    except IndexError:
        raise AssertionError(
            "Your regression has only one coefficient, so it has no constant: add one with "
            "sm.add_constant, as Step 5 does.") from None
    except Exception as e:
        raise AssertionError(
            f"coef_on_d(y, d, []) stopped with an error: {e}. With an empty list it should regress "
            "y on d alone. controls is a list of columns: make one list with d first, then the "
            "controls, rather than putting one list inside another.") from None
    assert none_yours is not None, "coef_on_d gave back nothing. Does it end with a return line?"
    # np.ndim is 0 for a single number
    assert np.ndim(none_yours) == 0, (
        "coef_on_d gives back several numbers. Give back D's coefficient alone, from .params.")
    assert isinstance(none_yours, (int, float, np.floating)), (
        "coef_on_d gives back the whole regression (or text). Take D's coefficient from .params.")
    assert not np.isclose(none_yours, without_w.params[0]), (
        "That is the constant, params[0]. D's coefficient comes right after it.")
    assert not np.isclose(none_yours, without_w.pvalues[1]), (
        "That is D's p-value. Give back its coefficient, from .params.")
    assert not np.isclose(none_yours, without_w.tvalues[1]), (
        "That is D's t-statistic. Give back its coefficient, from .params.")
    assert not (np.isclose(none_yours, round(without_w.params[1], 3))
                and not np.isclose(none_yours, without_w.params[1])), (
        "Your number is rounded. Give back the full number; round only when you print.")

    # Doubling y must double the coefficient; adding 100 to y must leave it alone
    doubled = coef_on_d(2 * y, d, [])
    assert not np.isclose(doubled, none_yours), (
        "coef_on_d(2 * y, d, []) gives the same answer as coef_on_d(y, d, []). Does the "
        "function use its y and d arguments, or the notebook's y and d?")
    assert np.isclose(coef_on_d(y + 100, d, []), none_yours), (
        "Adding 100 to every y changes your coefficient, so the regression has no constant. "
        "Add one with sm.add_constant, as Step 5 does.")
    assert not np.isclose(doubled, none_yours / 2), (
        "Doubling y halves your coefficient, so y and d are swapped. sm.OLS takes the outcome first.")
    assert np.isclose(doubled, 2 * none_yours), (
        "Doubling y should double the coefficient on d, and yours does not. Check the sm.OLS line.")

    assert not np.isclose(none_yours, with_w.params[1]), (
        "With no controls you get Step 8's Y ~ D + W number. Use the controls argument, not w.")
    assert np.isclose(none_yours, without_w.params[1]), (
        f"coef_on_d(y, d, []) gives {none_yours:.4f}, and Step 8's Y ~ D gives "
        f"{without_w.params[1]:.4f}.")

    with_yours = coef_on_d(y, d, [w])
    assert not np.isclose(with_yours, none_yours), (
        "coef_on_d(y, d, [w]) gives the same as with no controls. Does it use its controls argument?")
    assert np.isclose(with_yours, with_w.params[1]), (
        f"coef_on_d(y, d, [w]) gives {with_yours:.4f}, and Step 8's Y ~ D + W gives "
        f"{with_w.params[1]:.4f}. params[1] is D's coefficient only when d is the first column "
        "after the constant.")
    both = coef_on_d(y, d, [w, u1])
    assert not np.isclose(both, with_yours), (
        "With W and u1 you get the same as with W alone. Does the function use every array in "
        "controls, or only the first?")

    assert "m_bias_blocked" in globals(), (
        "Your function works. Now call it on Step 8's arrays and store the result as m_bias_blocked.")
    mine = m_bias_blocked
    assert np.ndim(mine) == 0, "m_bias_blocked should be one number."
    assert not np.isclose(mine, with_w.params[1]), (
        "m_bias_blocked is the W-only estimate from Step 8. Control for u1 as well.")
    assert not np.isclose(mine, coef_on_d(y, d, [u1])), (
        "m_bias_blocked controls for u1 but not W. The question is what happens with both.")
    assert not np.isclose(mine, coef_on_d(y, d, [w, u2])), (
        "That controls for W and u2. The task asks for W and u1.")
    assert not np.isclose(mine, none_yours), "m_bias_blocked has no controls in it. Add W and u1."
    assert not (np.isclose(mine, round(both, 3)) and not np.isclose(mine, both)), (
        "m_bias_blocked is rounded. Store the full number; round only when you print.")
    assert np.isclose(mine, both), (
        f"m_bias_blocked is {mine:.4f}, but coef_on_d with W and u1 as the controls gives {both:.4f}. "
        "Control for those two, and nothing else.")
    print("Passed. coef_on_d matches both of Step 8's estimates, and uses every control it is given.")
    print(f"Y ~ D + W + u1: {mine:.3f}   (Y ~ D + W was {with_w.params[1]:.3f}; the truth is 0)")

*Your answer here (the path, u1's role on it, and why you could not do this with LaLonde):*

---

## Extension (Optional — 15 min): Sample Size Does Not Cure Bias

Chapter 6's data defect identity said big samples cure noise, never bias. Here is the causal
version, and it is worth seeing the confidence interval tighten around the wrong number. The
simulation is World 2 from Part 1 at four sample sizes: the true effect of D on Y is 0.

In [ ]:
# EXTENSION — run as-is
# Watch precision improve while accuracy does not
print(f"{'n':>10}{'estimate':>12}{'95% CI width':>15}{'covers truth?':>16}")
print("-" * 55)

for size in [500, 5_000, 50_000, 500_000]:
    uu = rng.normal(0, 1, size)
    dd = 1.4 * uu + rng.normal(0, 1, size)
    yy = 1.4 * uu + rng.normal(0, 1, size)      # no dd term: the TRUE effect of D on Y is 0
    fit = sm.OLS(yy, sm.add_constant(dd)).fit()
    # conf_int() has one row per coefficient; row 1 is D's lower and upper limit
    lo, hi = fit.conf_int()[1]
    # lo <= 0 <= hi is True when the interval contains 0; str() lets it be padded like text
    print(f"{size:>10,}{fit.params[1]:>12.3f}{hi - lo:>15.3f}{str(lo <= 0 <= hi):>16}")

**Expected output:** at n = 500 the estimate is 0.707 and the interval is 0.136 wide. At
n = 500,000 the estimate is 0.664 and the interval is 0.004 wide: about 34 times narrower,
close to the square root of 1,000, for 1,000 times the data. It never once contains the true
effect of 0: the interval gets narrower around 0.66, not closer to 0.

**Try it.** Weaken the confounder: change **both** 1.4s, in the `dd` line and the `yy` line, to 0.2,
and run the cell again. Report the first n at which the interval stops covering the truth, and say
in one sentence why weakening the confounder delays the problem without ever fixing it. Run in
order, the first miss comes at n = 5,000; your n can differ if you ran cells out of order.

*Your answer here:*

---

## Challenge (Take-Home): Let the Graph Choose the Controls

You have been deciding by hand which variables to adjust for. That decision is mechanical
once the graph is written down — `networkx` can walk the paths for you. There is no starter code:
write your answer in the cell below and in new cells under it.

**Find a valid adjustment set from a DAG.**

1. Build this graph in `networkx`: U → D, U → Y, D → M → Y, D → C ← Y, and the M-bias structure
   from Step 8: U1 → W ← U2, with U1 → D and U2 → Y. Finally, D → Y.
2. List every path between D and Y in the *undirected* version of the graph, and classify each as
   causal (every arrow points away from D, towards Y), backdoor (it starts with an arrow into D),
   or non-causal and starting out of D (blocked unless you condition on C).
3. For a candidate adjustment set S, decide whether each backdoor path is blocked. A path is
   blocked by S if it contains a non-collider that is in S, or a collider such that neither it nor
   any of its descendants is in S.
4. Report the smallest valid S. Check it against Section 10.3's answer.
5. Reflect: `networkx` found the set mechanically from the graph. What part of this problem did it
   NOT solve, and why is that the hard part?

In [ ]:
# YOUR TASK — take-home Challenge: write your code here, and in new cells below

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Simulated three worlds (true causation, pure confounding and pure
  selection) whose correlations all come out near 0.65, although the
  true effects are 2.0, 0.0 and 0.0
* Drew a confounder, a collider and a mediator as directed graphs
  with networkx
* Showed that controlling for a confounder recovers the true effect,
  while controlling for a collider creates a large, highly
  significant association between two independent variables
* Built an M-bias structure in which a control measured BEFORE
  treatment still biases the estimate
* Wrote a small function, coef_on_d(), that returns the coefficient
  on the treatment for any list of controls, and used it to show that
  also controlling for one of the hidden causes closes the path again
* Reproduced the LaLonde (1986) contrast: the randomised experiment
  against a comparison group from the CPS, then measured how much of
  the gap covariate adjustment closes
* Key findings: correlations [YOUR VALUE], [YOUR VALUE] and
  [YOUR VALUE]; the collider control gave [YOUR VALUE] where the truth
  is 0; M-bias gave [YOUR VALUE], and [YOUR VALUE] once I also
  controlled for u1; LaLonde: experimental $[YOUR VALUE], naive CPS
  $[YOUR VALUE], with controls $[YOUR VALUE] ([YOUR VALUE]% of the gap
  closed)

**Please write a README.md entry including:**
1. Project Title: Correlation & Causality: Confounders, Colliders and the Limits of Controls
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab10-causal-logic`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab10-causal-logic`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab10-causal-logic`, branch `main`,
   commit message `Lab 10 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.